In [ ]:
# CELL 1: Setup & Imports (12 lines)
import numpy as np
import tensorflow as tf
import cv2
import logging
from violations.ml.config import PipelineConfig
from violations.ml.features import FeatureExtractor
from violations.ml.detection import LitterDetectionPipeline
from violations.ml.models import build_model
from violations.ml.evaluation import evaluate_model

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

CONFIG = PipelineConfig()

# CELL 2: Load & Prepare Data (20 lines)
from violations.ml.data import load_dataset

X_train, y_train, X_test, y_test = load_dataset(CONFIG.dataset_path)
logger.info(f"Loaded: Train {X_train.shape}, Test {X_test.shape}")

# CELL 3: Build Model (15 lines)
model = build_model(
    input_shape=(CONFIG.window_size_frames, CONFIG.feature_dimension),
    lstm_units=CONFIG.lstm_hidden_units,
    dropout=CONFIG.dropout_rate
)
model.compile(optimizer=tf.keras.optimizers.Adam(lr=CONFIG.learning_rate),
              loss='binary_crossentropy',
              metrics=['accuracy', tf.keras.metrics.Precision(), tf.keras.metrics.Recall()])

# CELL 4: Train Model (10 lines)
history = model.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=CONFIG.epochs,
    batch_size=CONFIG.batch_size,
    callbacks=[
        tf.keras.callbacks.EarlyStopping(patience=CONFIG.early_stopping_patience),
        tf.keras.callbacks.ModelCheckpoint('best_model.h5', save_best_only=True)
    ]
)

# CELL 5: Evaluate (25 lines)
metrics = evaluate_model(model, X_test, y_test)
logger.info(f"Accuracy: {metrics['accuracy']:.2%}")
logger.info(f"Precision: {metrics['precision']:.2%}")
logger.info(f"Recall: {metrics['recall']:.2%}")
logger.info(f"F1: {metrics['f1']:.2%}")

# Visualization (ROC, confusion matrix, etc.)

# CELL 6: Inference Demo (30 lines)
inference_engine = InferenceEngine('best_model.h5', CONFIG)
violations = inference_engine.process_video('test_video.mp4')

for v in violations:
    print(f"Violation: Person {v['person_id']}, Conf: {v['confidence']:.2%}")